In [1]:
%matplotlib inline
# Cell 1 — WO-02 Step 3, family 9: Band T. Load the Band T queries (read-only; the API is never called). No population sample has Band T.
import sys
from pathlib import Path
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'scripts' / 'edop').exists())
sys.path.insert(0, str(ROOT))
from scripts.edop.dashboard import wo02_payloads as P, wo02_rules_bandt as T, wo02_analysis as A

qs = T.bandt_queries()
BY_ID = {q['id']: q for q in qs}
print(f'Band T queries: {len(qs)} (the main fixtures with a span, plus F35 to F45)')
print([q['id'] for q in qs])


Band T queries: 18 (the main fixtures with a span, plus F35 to F45)
['F26', 'F27', 'F28', 'F29', 'F30', 'F33', 'F34', 'F35', 'F36', 'F37', 'F38', 'F39', 'F40', 'F41', 'F42', 'F43', 'F44', 'F45']


In [2]:
# Cell 2 — The ten Band T statement types (rules in scripts/edop/dashboard/wo02_rules_bandt.py)
for s in T.STATEMENTS:
    print(f"{s['id']}  {s['name']}")
    print(f"      applies to: {s['applies_to']}")
    print(f"      template  : {s['template']}")
print()
print('labels for the LMR and HYDE variables come from the rules (Band T rows have no friendly name); record bounds: LMR', T.LMR_FIRST, 'to', T.LMR_LAST, '| eVolv2k', T.EV_FIRST, 'to', T.EV_LAST)


BT-01  LMR anomaly over the span, per variable
      applies to: single basin, buffer, polity; air temperature, precipitation rate, PDSI; spans inside 0 to 1998 CE
      template  : {name}, {from} to {to}: the yearly anomaly averages {n} {units}, lowest {n} in {year}, highest {n} in {year}, against the 850-1850 CE reference. | In {year} the anomaly is {n} ...
BT-02  LMR spread across grid cells (middle year)
      applies to: single basin, buffer, polity; LMR variables
      template  : {name}: in {year}, the values across the {n} grid cells range from {n} to {n} {units}.
BT-03  How many LMR grid cells the footprint touches
      applies to: single basin, buffer, polity
      template  : The footprint touches {n} LMR grid cells (each 2 degrees wide), so each yearly value is built from {n} cells; the engine marks this low resolution.
BT-04  The span runs outside the LMR record
      applies to: single basin, buffer, polity
      template  : LMR covers {from} to {to} of the span {from} t

In [3]:
# Cell 3 — Apply every rule to every Band T query; count where each applies and why not
df = A.apply_fixtures(qs, T)
print(A.fixture_table(df).to_string())
print()
print('why a statement did not apply (count of items):')
print(A.reason_table(df).to_string())


res                 applied  not_applicable
id    scope                                
BT-01 buffer              9               0
      polity              6               3
      single_basin       33               3
BT-02 buffer              9               0
      polity              6               3
      single_basin       33               3
BT-03 buffer              3               0
      polity              2               1
      single_basin       11               1
BT-04 buffer              0               3
      polity              1               2
      single_basin        3               9
BT-05 buffer              2               1
      polity              3               0
      single_basin        8               4
BT-06 buffer              4               8
      polity              0              12
      single_basin       16              32
BT-07 buffer              0               3
      polity              1               2
      single_basin        3     

In [4]:
# Cell 4 — Draft sentences on chosen queries, to read and judge
for qid in ['F26', 'F29', 'F34', 'F39', 'F40', 'F41', 'F42', 'F45']:
    q = BY_ID[qid]
    g = df[(df.qid == qid) & df.applies]
    print(f"--- {qid}: {q.get('condition')}")
    for _, r in g.iterrows():
        print(f"  {r['id']}  {r['text']}")
    print()


--- F26: CE span with LMR and HYDE; 7 eVolv2k events
  BT-01  LMR air temperature, 1475 to 1525: the yearly anomaly averages -0.17 K, lowest -0.35 K in 1525, highest +0.031 K in 1517, against the 850-1850 CE reference.
  BT-02  LMR air temperature: in 1500, the 3 grid cells agree: -0.16 K.
  BT-01  LMR precipitation rate, 1475 to 1525: the yearly anomaly averages -0.013 mm/day, lowest -0.052 mm/day in 1498, highest +0.023 mm/day in 1510, against the 850-1850 CE reference.
  BT-02  LMR precipitation rate: in 1500, the values across the 3 grid cells range from -0.023 to -0.0082 mm/day.
  BT-01  LMR drought index (PDSI), 1475 to 1525: the yearly anomaly averages -0.089, lowest -0.4 in 1514, highest +0.11 in 1478, against the 850-1850 CE reference.
  BT-02  LMR drought index (PDSI): in 1500, the values across the 3 grid cells range from -0.12 to -0.021.
  BT-03  The footprint touches 3 LMR grid cells (each 2 degrees wide), so each yearly value is built from 3 cells; the engine marks this l

In [5]:
# Cell 5 — What the evidence is: what each Band T query carries
inv = pd.DataFrame(T.inventory(qs))
print(inv.to_string(index=False))
print()
print('scopes:', inv.scope.value_counts().to_dict(), '| levels:', inv.level.value_counts().to_dict())


qid        scope  level         span       lmr_years  lmr_cells low_res  hyde_epochs  hyde_n_cells  events                      caveats  rows   kb
F26 single_basin      6 1475 to 1525  1475-1525 (51)        3.0    True            1            80       7                   lmr_caveat   164  179
F27 single_basin      6 1250 to 1300  1250-1300 (51)        3.0    True            1            80       5                   lmr_caveat   162  178
F28 single_basin      6 -200 to -100            none        NaN    None            1           350      12            hyde_nearest_year    16   51
F29       polity      6  961 to 1027   961-1027 (67)       39.0   False            1         13147       8                   lmr_caveat   213  202
F30 single_basin      6 1000 to 1100 1000-1100 (101)        5.0    True            2           282       9                   lmr_caveat   320  309
F33 single_basin      6 1140 to 1170  1140-1170 (31)        3.0    True            1            80       0 hyde_neares

In [6]:
# Cell 6 — The numbers inside the statements (fixtures only; no weighting, no population)
b1 = df[(df.id == 'BT-01') & df.applies].copy()
b1['mean'] = b1.stat.astype(float)
print('BT-01 mean yearly LMR anomaly per query, by variable (count, min, median, max across the queries):')
print(b1.groupby('variable')['mean'].agg(['count', 'min', 'median', 'max']).round(3).to_string())
print()
b2 = df[(df.id == 'BT-02') & df.applies].copy()
b2['spread'] = b2.stat.astype(float)
print('BT-02 spread across grid cells in the middle year (highest minus lowest cell), by variable:')
print(b2.groupby('variable')['spread'].agg(['count', 'min', 'median', 'max']).round(3).to_string())
print()
print('BT-03 grid cells touched, by query:')
print(df[(df.id == 'BT-03') & df.applies][['qid', 'scope', 'level', 'stat']].rename(columns={'stat': 'cells'}).to_string(index=False))
print()
b6 = df[(df.id == 'BT-06') & df.applies][['qid', 'variable', 'stat']].rename(columns={'stat': 'share_change'})
print('BT-06 HYDE change across epochs as a share of the first epoch (blank where the first is zero):')
print(b6.to_string(index=False))


BT-01 mean yearly LMR anomaly per query, by variable (count, min, median, max across the queries):
           count    min  median    max
variable                              
lmr_air       16 -0.363  -0.162  0.318
lmr_pdsi      16 -0.206   0.014  0.431
lmr_prate     16 -0.015  -0.001  0.030

BT-02 spread across grid cells in the middle year (highest minus lowest cell), by variable:
           count    min  median    max
variable                              
lmr_air       16  0.004   0.035  0.287
lmr_pdsi      16  0.073   0.101  0.826
lmr_prate     16  0.005   0.014  0.131

BT-03 grid cells touched, by query:
qid        scope  level  cells
F26 single_basin      6    3.0
F27 single_basin      6    3.0
F29       polity      6   39.0
F30 single_basin      6    5.0
F33 single_basin      6    3.0
F34 single_basin      6    3.0
F35       buffer      6    4.0
F36       buffer      6    9.0
F37 single_basin      8    2.0
F38       polity      8   39.0
F40 single_basin      6    3.0
F41 singl

In [7]:
# Cell 7 — Cutoffs in this family: only one exists in the Sandbox (events of 5 Tg S or more are shown first). Grid over event size
(lmr_span, events, epochs) = T.record_bounds(qs)
sz = pd.Series([s for _, s in events])
print(f'distinct volcanic events seen in the Band T payloads: {len(sz)} (years {events[0][0]} to {events[-1][0]})')
print('share of events at least each size (Tg S):', {f'>= {t}': round(float((sz >= t).mean()), 3) for t in [1, 2, 5, 10, 20]})
print('size quantiles (Tg S):', {f'p{int(q*100)}': round(float(sz.quantile(q)), 2) for q in (0.1, 0.25, 0.5, 0.75, 0.9)}, '| largest:', max(events, key=lambda e: e[1]))
print()
print('the BT-09 sentence states the count and the largest event, so it carries no size cutoff')
print('the low_resolution flag (engine threshold: effective weight under 5) cannot be swept: LMR rows do not carry the effective weight')


distinct volcanic events seen in the Band T payloads: 79 (years -190 to 1512)
share of events at least each size (Tg S): {'>= 1': 0.747, '>= 2': 0.494, '>= 5': 0.241, '>= 10': 0.139, '>= 20': 0.025}
size quantiles (Tg S): {'p10': 0.68, 'p25': 0.98, 'p50': 1.97, 'p75': 4.52, 'p90': 11.68} | largest: (1257, 59.42)

the BT-09 sentence states the count and the largest event, so it carries no size cutoff
the low_resolution flag (engine threshold: effective weight under 5) cannot be swept: LMR rows do not carry the effective weight


In [8]:
# Cell 8 — Checks: record bounds, partial coverage, the flag, the HYDE totals
print('LMR years seen across the payloads:', lmr_span, '(the record bounds assumed by BT-04: 0 to 1998)')
print('HYDE epochs seen (%d):' % len(epochs), epochs[:9], '...', epochs[-3:])
print('  cadence: centennial and similar before 1950, annual from 1950 on (the payload has no epoch list)')
print()
print('queries where the span runs outside the LMR record (BT-04):')
print(df[(df.id == 'BT-04') & df.applies][['qid', 'text']].to_string(index=False))
print()
print('low_resolution by query (LMR cells, flag):')
print(inv[inv.lmr_cells.notna()][['qid', 'scope', 'level', 'lmr_cells', 'low_res']].to_string(index=False))
print()
n, eq, worst = T.hyde_grazing_check(qs)
print(f'HYDE: grazing land equals pasture + rangeland (within 0.5 km2) in {eq} of {n} epochs; worst difference {worst:,.0f} km2 (grazing is always the larger)')
print()
print('payload size by span length:')
print(inv[['qid', 'span', 'rows', 'kb']].sort_values('rows').to_string(index=False))


LMR years seen across the payloads: (0, 1998) (the record bounds assumed by BT-04: 0 to 1998)
HYDE epochs seen (61): [0, 800, 900, 1000, 1100, 1200, 1300, 1500, 1940] ... [2018, 2019, 2020]
  cadence: centennial and similar before 1950, annual from 1950 on (the payload has no epoch list)

queries where the span runs outside the LMR record (BT-04):
qid                                                                                                               text
F28                                   The span lies outside the LMR record (0 to 1998 CE), so there are no LMR values.
F39                                   The span lies outside the LMR record (0 to 1998 CE), so there are no LMR values.
F41 LMR covers 1980 to 1998 of the span 1980 to 2020 (the record runs 0 to 1998 CE); there are no values for the rest.
F42   LMR covers 0 to 50 of the span 50 BCE to 50 CE (the record runs 0 to 1998 CE); there are no values for the rest.

low_resolution by query (LMR cells, flag):
qid        

In [9]:
# Cell 9 — Write this family's rows to the master table (replaces any earlier Band T rows)
fxt = A.fixture_table(df)
rows = []
for s in T.STATEMENTS:
    sid = s['id']
    f = fxt.loc[sid]
    row = dict(s, family='Band T')
    row['fixture_result'] = f"{int(f.applied.sum())} items applied, {int(f.not_applicable.sum())} not applicable ({len(qs)} Band T queries)"
    row['population_result'] = 'not sampled: the population sample has no Band T; the evidence is the Band T fixtures only'
    row['sweep_result'] = ('events shown from 5 Tg S is a Sandbox display cutoff, swept in Cell 7; the sentence states the count and the largest' if sid == 'BT-09'
                           else 'low_resolution cannot be swept (effective weight is not in the LMR rows)' if sid == 'BT-03' else 'no cutoff')
    rows.append(row)
n_rows = P.upsert_master('Band T', rows)
print('master table now has', n_rows, 'rows ->', P.MASTER)


master table now has 54 rows -> /Users/karlg/Documents/repos/_edops/notebooks/edop/dashboard/wo02_master_table.csv
